In [1]:
import os
os.chdir(r"E:\text_summarizer")
print(os.getcwd())

E:\text_summarizer


In [2]:
from textSummarizer.entity import ModelTrainingConfig
print(ModelTrainingConfig)


<class 'textSummarizer.entity.ModelTrainingConfig'>


In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH):
    
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_model_trainer_config(self) -> ModelTrainingConfig:
        config = self.config.model_trainer
        params = self.params.model_trainer_args
        
        model_training_config = ModelTrainingConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        
        return model_training_config

In [6]:
from transformers import TrainingArguments, Trainer, Seq2SeqTrainingArguments, Seq2SeqTrainer
from textSummarizer.config.configuration import ConfigurationManager, ModelTrainingConfig
from transformers import DataCollatorForSeq2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset, load_from_disk
import torch

e:\text_summarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class ModelTrainer:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config
        
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        
        dataset_samsum_pt["train"] = dataset_samsum_pt["train"].select(range(500))
        dataset_samsum_pt["validation"] = dataset_samsum_pt["validation"].select(range(100))

        
        training_args = Seq2SeqTrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            predict_with_generate=True,
            gradient_checkpointing=True
        )
        
        trainer = Seq2SeqTrainer(
            model=model_pegasus,
            args=training_args,
            processing_class=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
            
        )
        
        trainer.train()    
        
        #Save the trained model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "T5_Small"))
        #save Tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "Tokenizer"))
        

In [8]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-02-18 08:37:36,121: INFO: textSummarizer: YAML file: E:\text_summarizer\config\config.yaml loaded successfully]
[2026-02-18 08:37:36,126: INFO: textSummarizer: YAML file: E:\text_summarizer\params.yaml loaded successfully]
[2026-02-18 08:37:36,129: INFO: textSummarizer: created directory at: artifacts]
[2026-02-18 08:37:36,131: INFO: textSummarizer: created directory at: artifacts/model_trainer]
[2026-02-18 08:37:36,510: INFO: httpx: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-02-18 08:37:36,792: INFO: httpx: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-02-18 08:37:37,131: INFO: httpx: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-02-18 08:37:37,392: INFO: httpx: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/ma

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 347.97it/s, Materializing param=shared.weight]                                                      


[2026-02-18 08:37:40,208: INFO: httpx: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]


e:\text_summarizer\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.34it/s]
